# Laboratorium 2: Współbieżność i Równoległość w Pythonie
### Skoroszyt Edukacyjny - Wersja dla Studentów

---

## 1. Wstęp: Koncepcja "Wielu Zadań"

Zanim zaczniemy pisać kod, musimy rozróżnić dwa kluczowe pojęcia:

1. **Współbieżność (Concurrency)**: Wykonywanie wielu zadań "na zmianę". Wyobraź sobie kelnera, który obsługuje 5 stolików. Nie robi wszystkiego naraz, ale szybko przełącza się między nimi. Dla klientów wygląda to, jakby obsługiwał ich równocześnie.
2. **Równoległość (Parallelism)**: Wykonywanie wielu zadań faktycznie w tym samym momencie. To sytuacja, w której mamy 5 kelnerów i każdy obsługuje jeden stolik.

W Pythonie współbieżność realizujemy najczęściej za pomocą **Wątków (Threads)**, a równoległość za pomocą **Procesów (Processes)**.

---

## 2. Wielowątkowość (Threading) - Zadania I/O-bound

Wątki są idealne, gdy program większość czasu spędza na **czekaniu** na odpowiedź z sieci (zapytania HTTP). W tym czasie procesor się nudzi – wątki pozwalają mu wysłać kolejne zapytania, nie czekając na poprzednie.

---

### Demo: Scraping Kalendarza Kulturalnego (Krakow.pl)

**Kod zawarty w poniższych komórkach (analogicznie do plików `lab_2_1_demo.py` oraz `lab_2_2_demo.py`) pozwala na pobieranie tytułów wydarzeń kulturalnych z oficjalnego kalendarium miasta Krakowa (krakow.pl).** 

Przykładowy adres źródłowy: `https://www.krakow.pl/kalendarium/1919,shw,2026-03-20,0,day.html`. 

Demo pokazuje proces pobierania danych z 5 kolejnych stron tego zestawienia:
1. **Wersja sekwencyjna**: Zadanie wykonywane jest krok po kroku, co pozwala zaobserwować sumaryczny czas oczekiwania na każde z zapytań HTTP z osobna (wysoki koszt operacji wejścia/wyjścia).
2. **Optymalizacja**: Kod zostaje zmodyfikowany z użyciem modułu `concurrent.futures`, wykorzystując `ThreadPoolExecutor`.

Dzięki temu zapytania sieciowe są wysyłane równolegle, co drastycznie skraca czas całkowity działania programu, demonstrując praktyczną przewagę wielowątkowości w zadaniach typu **I/O-bound** (zależnych od odpowiedzi sieciowej).

In [2]:
import requests
from bs4 import BeautifulSoup
import time

def download_site(url):
    """Pobiera jedną stronę i wyciąga tytuły wydarzeń."""
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    event_titles = [item.text.strip() for item in soup.select('.item__link h3')]
    return event_titles

def run_sequential_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie SEKWENCYJNE 5 stron...")
    start = time.time()
    
    all_titles = []
    for url in sites:
        all_titles.extend(download_site(url))
        
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania: {time.time() - start:.2f}s")

run_sequential_demo()

Rozpoczynam pobieranie SEKWENCYJNE 5 stron...
Pobrano łącznie 100 tytułów.
Pierwsze 10 wyników:
1. Bertolt Brecht. Uczycie nas, jak suknie się podnosi
2. Gdzie gubią się zdania
3. [ODWOŁANY] Sąsiadka
4. #Osiecka
5. Italian Passion – Burlesque Night
6. Wielki Gatsby
7. Niezależna Białoruś: Biaz Nazvy, Syndrom Samazvanca, Atesta
8. Wiosenne Triduum Organowe
9. Wszystko, co najlepsze
10. Czyż nie dobija się koni?

Czas wykonania: 3.89s


In [3]:
import concurrent.futures

def run_threaded_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...")
    start = time.time()
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
        results = list(executor.map(download_site, sites))
    
    all_titles = [title for sublist in results for title in sublist]
    
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania (wątki): {time.time() - start:.2f}s")

run_threaded_demo()

Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...
Pobrano łącznie 100 tytułów.
Pierwsze 10 wyników:
1. Bertolt Brecht. Uczycie nas, jak suknie się podnosi
2. Gdzie gubią się zdania
3. [ODWOŁANY] Sąsiadka
4. #Osiecka
5. Italian Passion – Burlesque Night
6. Wielki Gatsby
7. Niezależna Białoruś: Biaz Nazvy, Syndrom Samazvanca, Atesta
8. Wiosenne Triduum Organowe
9. Wszystko, co najlepsze
10. Czyż nie dobija się koni?

Czas wykonania (wątki): 0.95s


--- 
## 3. Synchronizacja: Problem Hazardu i Lock

Gdy wiele wątków próbuje zmieniać tę samą zmienną w tym samym momencie (np. saldo na koncie), dochodzi do tzw. **Race Condition** (wyścigu). Rozwiązaniem jest **Lock** (blokada).

In [4]:
import threading

class BankAccount:
    def __init__(self):
        self.balance = 0
        self.lock = threading.Lock()

    def deposit(self, amount):
        with self.lock:
            current = self.balance
            time.sleep(0.0001) # Symulacja opóźnienia
            self.balance = current + amount

account = BankAccount()
with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    executor.map(lambda _: account.deposit(1), range(100))
    
print(f"Saldo końcowe: {account.balance} zł (oczekiwano: 100)")

Saldo końcowe: 100 zł (oczekiwano: 100)


--- 
## 4. Wieloprocesowość (Multiprocessing) - Zadania CPU-bound

Kiedy musimy wykonać ciężkie obliczenia matematyczne (np. szukanie liczb pierwszych), wątki nam nie pomogą. Musimy użyć osobnych procesów.

**Ważne (macOS/Windows)**: Ze względu na metodę `spawn` startu procesów, funkcje pomocnicze (jak `find_primes`) muszą znajdować się w zewnętrznym pliku `.py` (tutaj: `lab2_functions.py`) i być importowane.

In [5]:
import multiprocessing
import time
# Importujemy funkcję z oddzielnego pliku, aby uniknąć błędu spawn na macOS
from lab2_functions import find_primes

def run_primes_demo():
    cores = multiprocessing.cpu_count()
    print(f"Praca na {cores} procesach (rdzeniach)...")
    start = time.time()
    
    limit = 1_000_000
    chunk = limit // cores
    ranges = [(i, i + chunk) for i in range(0, limit, chunk)]

    with multiprocessing.Pool(processes=cores) as pool:
        results = pool.starmap(find_primes, ranges)
    
    print(f"Zakończono w czasie {time.time() - start:.2f}s.")

if __name__ == "__main__":
    run_primes_demo()

Praca na 24 procesach (rdzeniach)...
Zakończono w czasie 0.32s.


---
# Zadania do samodzielnego wykonania

Poniższe zadania należy zrealizować w oparciu o wiedzę zdobytą na laboratoriach oraz instrukcje zawarte w pliku PDF.

### Zadanie 1 (Threading)
Przy użyciu publicznego API **Cat Facts** (`https://catfact.ninja/fact`), które zwraca przy każdym wywołaniu losowy fakt na temat kotów:
1. Pobierz sekwencyjnie 20 faktów i zmierz czas całkowitego działania programu.
2. Zmodyfikuj kod, aby wysyłać zapytania wielowątkowo przy użyciu `ThreadPoolExecutor`.
3. Porównaj czasy wykonania.

*Podpowiedź: Użyj `requests.get(URL).json().get('fact')`*

In [ ]:
# Miejsce na rozwiązanie zadania 1
import requests
import time
import concurrent.futures

CAT_API_URL = "https://catfact.ninja/fact"

def get_cat_fact():
    response = requests.get(CAT_API_URL)
    return response.json().get("fact")


# 1. Pobieranie sekwencyjne
def fetch_sequential(n=20):
    facts = []
    start = time.time()

    for _ in range(n):
        fact = get_cat_fact()
        facts.append(fact)

    end = time.time()
    duration = end - start

    return facts, duration


# 2. Pobieranie wielowątkowe
def fetch_threaded(n=20):
    start = time.time()

    with concurrent.futures.ThreadPoolExecutor() as executor:
        facts = list(executor.map(lambda _: get_cat_fact(), range(n)))

    end = time.time()
    duration = end - start

    return facts, duration



sequential_facts, sequential_time = fetch_sequential()
threaded_facts, threaded_time = fetch_threaded()

print("=== Sekwencyjnie ===")
for i, fact in enumerate(sequential_facts, 1):
    print(f"{i}. {fact}")
print(f"Czas sekwencyjny: {sequential_time:.4f} s\n")

print("=== Wielowątkowo ===")
for i, fact in enumerate(threaded_facts, 1):
    print(f"{i}. {fact}")
print(f"Czas wielowątkowy: {threaded_time:.4f} s\n")

print("=== Porównanie ===")
if threaded_time < sequential_time:
    print("Wersja wielowątkowa była szybsza.")
else:
    print("Wersja sekwencyjna była szybsza.")

=== Sekwencyjnie ===
1. All cats need taurine in their diet to avoid blindness. Cats must also have fat in their diet as they are unable to produce it on their own.
2. Ancient Egyptian family members shaved their eyebrows in mourning when the family cat died.
3. Among many other diseases, cats can suffer from anorexia, senility, feline AIDS and acne.
4. The claws on the cat’s back paws aren’t as sharp as the claws on the front paws because the claws in the back don’t retract and, consequently, become worn.
5. The smallest pedigreed cat is a Singapura, which can weigh just 4 lbs (1.8 kg), or about five large cans of cat food. The largest pedigreed cats are Maine Coon cats, which can weigh 25 lbs (11.3 kg), or nearly twice as much as an average cat weighs.
6. Cats can be right-pawed or left-pawed.
7. The largest breed of cat is the Ragdoll with males weighing in at 1 5 to 20 lbs. The heaviest domestic cat on record was a neutered male tabby named Himmy from Queensland, Australia who weig

### Zadanie 2 (Wątki i Kolejka - Producent-Konsument)
Napisz program o strukturze **producent-consumers**:
1. **Producent**: Generuje kolejne liczby naturalne i dodaje je do kolejki (`queue.Queue`).
2. **Konsument 1**: Pobiera z kolejki tylko liczby **parzyste**.
3. **Konsument 2**: Pobiera z kolejki tylko liczby **nieparzyste**.

Użyj wątków do realizacji producenta i obu konsumentów. Program powinien zakończyć się po przetworzeniu określonej puli liczb.

In [7]:
# Miejsce na rozwiązanie zadania 2
import queue
import threading
import time

q = queue.Queue()


def producent(limit):
    for i in range(1, limit + 1):
        print(f"Producent dodał: {i}")
        q.put(i)
        time.sleep(0.1)

    q.put(None)
    q.put(None)


def konsument_parzyste():
    while True:
        item = q.get()

        if item is None:
            break

        if item % 2 == 0:
            print(f"Konsument parzyste pobrał: {item}")
        else:
            q.put(item)

        time.sleep(0.1)


def konsument_nieparzyste():
    while True:
        item = q.get()

        if item is None:
            break

        if item % 2 != 0:
            print(f"Konsument nieparzyste pobrał: {item}")
        else:
            q.put(item)

        time.sleep(0.1)


limit = 10

producer_thread = threading.Thread(target=producent, args=(limit,))
even_thread = threading.Thread(target=konsument_parzyste)
odd_thread = threading.Thread(target=konsument_nieparzyste)

producer_thread.start()
even_thread.start()
odd_thread.start()

producer_thread.join()
even_thread.join()
odd_thread.join()

print("Wszystkie liczby zostały przetworzone.")

Producent dodał: 1
Konsument nieparzyste pobrał: 1
Producent dodał: 2
Konsument parzyste pobrał: 2
Producent dodał: 3
Konsument nieparzyste pobrał: 3
Producent dodał: 4
Konsument parzyste pobrał: 4
Producent dodał: 5
Konsument nieparzyste pobrał: 5
Producent dodał: 6
Konsument parzyste pobrał: 6
Producent dodał: 7
Konsument nieparzyste pobrał: 7
Producent dodał: 8
Konsument parzyste pobrał: 8
Producent dodał: 9
Konsument nieparzyste pobrał: 9
Producent dodał: 10
Konsument parzyste pobrał: 10
Wszystkie liczby zostały przetworzone.


### Zadanie 3 (Multiprocessing)
Napisz program, który zrównolegli obliczanie sumy kolejnych stu potęg dla każdej liczby z ciągu liczb naturalnych w dużym zakresie (np. 1 - 10 000).
Użyj modułu `multiprocessing` oraz gotowej funkcji `calculate_power_sum(n)` z pliku `lab2_functions.py`.

Pamiętaj o bezpiecznym uruchamianiu procesów na macOS (`if __name__ == "__main__":`).

In [8]:
# Miejsce na rozwiązanie zadania 3
import multiprocessing
import time
from lab2_functions import calculate_power_sum

if __name__ == "__main__":
    numbers = list(range(1, 10001))

    start = time.time()

    with multiprocessing.Pool() as pool:
        results = pool.map(calculate_power_sum, numbers)

    end = time.time()

    print("Pierwsze 10 wyników:")
    for i, result in enumerate(results[:10], 1):
        print(f"{i}: {result}")

    print(f"Czas wykonania: {end - start:.4f} s")

Pierwsze 10 wyników:
1: 100
2: 2535301200456458802993406410750
3: 773066281098016996554691694648431909053161283000
4: 2142584059011987034055949456454883470029603991710390447068500
5: 9860761315262647567646607066034827870915080438862787559628486633300780
6: 783982348200085087316028320589669384644572452567545845851686359643396569772850
7: 3773555927895550989902089063950252946000070398722062967756211219956973369576416070000
8: 2328041115810841241449652215325003612630249592761070000727017656405007199729527664209597000
9: 298815737485359115506128987290252080182887634235068807971396831956479052263964955868682786424500
10: 11111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111110
Czas wykonania: 0.1572 s
